# One Layer Cannot Do XOR, and Two Can

**Topic:** Multi-layer perceptrons &nbsp;·&nbsp; **Difficulty:** Medium

The XOR problem is the reason neural networks have hidden layers. No straight
line separates the classes, so a linear model is stuck near chance no matter how
long you train it, and adding one hidden layer of four units fixes it
almost completely.

Train both from scratch and watch the boundary appear.

## The data (created in the next cell)

- `X` `(800, 2)`, `y` `(800,)`: the XOR pattern with Gaussian spread.
- `X_test` `(400, 2)`, `y_test` from the same generator.

## Your task

With **NumPy only**:

1. Train a **linear** logistic model by full-batch gradient descent: 3000 steps,
   learning rate 0.5. Record its test accuracy as `acc_linear`.
2. Train a 2 -> 4 -> 1 network with `tanh` hidden activation and a sigmoid
   output, on the mean binary cross-entropy loss, for 6000 full-batch steps at
   learning rate 0.5. Initialize with `np.random.default_rng(5)`:
   `W1 = rng.normal(0, 1, (2, 4))`, `b1 = zeros(4)`,
   `W2 = rng.normal(0, 1, (4, 1))`, `b2 = zeros(1)`.
3. Record `acc_mlp` and the final training loss `loss_mlp`.
4. Set `answers` to a dict with `"acc_linear"`, `"acc_mlp"`, `"loss_mlp"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the linear model is below 0.62 and the MLP is above 0.9.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    q = r.integers(0, 4, n)
    centres = np.array([[-1, -1], [1, 1], [-1, 1], [1, -1]], dtype=float)
    Z = centres[q] + r.normal(0, 0.42, (n, 2))
    lab = (q >= 2).astype(float)          # XOR: opposite corners share a class
    return Z, lab

X, y = make(800, 71)
X_test, y_test = make(400, 72)
print(f"X {X.shape}, class balance {y.mean():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_linear", [], 0.4425, -0.07409132155541588, 1e-06),
    ("acc_mlp", [], 0.9825, -0.16450784955524542, 1e-06),
    ("loss_mlp", [], 0.03769539497176942, -0.006311642101721525, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "acc_mlp" in answers and "acc_linear" in answers:
    _g = answers["acc_linear"] < 0.62 and answers["acc_mlp"] > 0.9
    print(f"  {'hidden layer is needed':<24} {'ok' if _g else 'wrong'}  "
          f"(linear {answers['acc_linear']:.3f}, mlp {answers['acc_mlp']:.3f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

XOR is the canonical non-linearly-separable problem: the two classes sit on
opposite diagonals, so any line puts one member of each class on both sides. The
hidden layer builds two new coordinates that *are* linearly separable. Each
`tanh` unit carves a half-plane, and the output layer combines them. This is
exactly the representational argument for depth, in the smallest example where
it bites.

```python
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

n = len(y)
w = np.zeros(2)
b = 0.0
for _ in range(3000):
    p = sigmoid(X @ w + b)
    g = X.T @ (p - y) / n
    gb = float((p - y).mean())
    w -= 0.5 * g
    b -= 0.5 * gb
acc_linear = float(((sigmoid(X_test @ w + b) > 0.5).astype(float) == y_test).mean())

rng = np.random.default_rng(5)
W1 = rng.normal(0, 1, (2, 4))
b1 = np.zeros(4)
W2 = rng.normal(0, 1, (4, 1))
b2 = np.zeros(1)

Y = y.reshape(-1, 1)
for _ in range(6000):
    H = np.tanh(X @ W1 + b1)
    P = sigmoid(H @ W2 + b2)
    dZ2 = (P - Y) / n
    gW2 = H.T @ dZ2
    gb2 = dZ2.sum(axis=0)
    dH = dZ2 @ W2.T
    dA = dH * (1 - H ** 2)
    gW1 = X.T @ dA
    gb1 = dA.sum(axis=0)
    W1 -= 0.5 * gW1
    b1 -= 0.5 * gb1
    W2 -= 0.5 * gW2
    b2 -= 0.5 * gb2

H = np.tanh(X @ W1 + b1)
P = sigmoid(H @ W2 + b2)
loss_mlp = float(-(Y * np.log(P + 1e-12) + (1 - Y) * np.log(1 - P + 1e-12)).mean())

Ht = np.tanh(X_test @ W1 + b1)
Pt = sigmoid(Ht @ W2 + b2)
acc_mlp = float(((Pt.ravel() > 0.5).astype(float) == y_test).mean())

answers = {"acc_linear": acc_linear, "acc_mlp": acc_mlp, "loss_mlp": loss_mlp}
```